# LLM Output Consistency Testing

**Same prompt, N runs — how consistent is your model?**

This notebook builds a reusable **consistency scorer** for LLM outputs. It runs one prompt
`N` times against a model (Groq or Gemini), then measures how stable the outputs are along
three axes:

1. **Semantic consistency** — embedding-based pairwise similarity between responses
2. **Structural consistency** — length variance, and agreement on the *final answer* (if the
   prompt has one, e.g. a math problem)
3. **Lexical diversity** — how much wording varies even when meaning stays similar

Built to slot straight into a Groq (`llama-3.1-8b-instant`) / Gemini (`gemini-2.5-flash-lite`)
workflow — same models used in CoT step-order perturbation-style experiments — so you can reuse
this as a sanity-check layer before running larger perturbation trials.

**Sections:** Install → Imports → API Config → Prompt → Generate → Save → Embeddings →
Similarity Matrix → Consistency Score → Structural Metrics → Diversity Metrics →
Visualizations → Final Report → Export


## 1. Install Libraries

In [1]:
%pip install -q groq google-generativeai python-dotenv


Note: you may need to restart the kernel to use updated packages.


## 2. Import necessary libraries


In [3]:
import os
import re
import json
import time
import math
import getpass
from datetime import datetime
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics.pairwise import cosine_similarity

sns.set_theme(style="whitegrid")


In [4]:
GROQ_API_KEY = os.environ.get("GROQ_API_KEY")
GEMINI_API_KEY = os.environ.get("GEMINI_API_KEY") 

groq_client = None
gemini_model_client = None

if GROQ_API_KEY:
    from groq import Groq
    groq_client = Groq(api_key=GROQ_API_KEY)

if GEMINI_API_KEY:
    import google.generativeai as genai
    genai.configure(api_key=GEMINI_API_KEY)

print("Groq configured:", groq_client is not None)
print("Gemini configured:", GEMINI_API_KEY != "")

Groq configured: True
Gemini configured: True
